In [1]:
import os
import pandas as pd

print(os.getcwd())

/Users/evannuss/Documents/retail_analytics_portfolio/notebooks


In [2]:
import pandas as pd

customers = pd.read_csv("../data/raw/customers.csv")
employees = pd.read_csv("../data/raw/employees.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
orders = pd.read_csv("../data/raw/orders.csv")
products = pd.read_csv("../data/raw/products.csv")
returns = pd.read_csv("../data/raw/returns.csv")
stores = pd.read_csv("../data/raw/stores.csv")


In [3]:
order_id_duplicates = orders.duplicated(subset="order_id", keep=False)

order_id_duplicates_rows = orders[order_id_duplicates]

print(order_id_duplicates_rows)


      order_id customer_id           order_date store_id employee_id  \
588    O000589      C06384  2025-03-28 04:00:15     S012       E0121   
1163   O001164      C04233  2025-09-07 18:58:30     S009       E0264   
1231   O001232      C07371  2024-10-31 20:42:45     S005         NaN   
1571   O001572      C03195  2024-04-15 01:13:40     S010         NaN   
3556   O003557      C07607  2025-11-19 06:02:00     S019       E0079   
...        ...         ...                  ...      ...         ...   
50095  O042573      C04232  2024-04-14 13:43:38     S021         NaN   
50096  O045107      C06483  2024-07-06 05:19:10     S005         NaN   
50097  O028133      C06605  2025-09-21 21:00:50     S025         NaN   
50098  O041453      C02652  2026-02-09 05:13:43     S013       E0077   
50099  O027747      C02957  2025-03-10 11:51:18     S015       E0130   

          channel payment_method order_status  
588    Mobile App      Gift Card    Completed  
1163        Store         PayPal    Com

In [4]:
print("Count: ", order_id_duplicates_rows.count())

Count:  order_id          200
customer_id       200
order_date        200
store_id          200
employee_id       118
channel           200
payment_method    200
order_status      200
dtype: int64


# Data Quality Assessment

# Customers table overview

In [6]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 8120 entries, 0 to 8119
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   customer_id    8120 non-null   str  
 1   first_name     8120 non-null   str  
 2   last_name      8120 non-null   str  
 3   email          7770 non-null   str  
 4   phone          7870 non-null   str  
 5   city           8120 non-null   str  
 6   state          8120 non-null   str  
 7   region         8120 non-null   str  
 8   customer_tier  8120 non-null   str  
 9   signup_date    8120 non-null   str  
dtypes: str(10)
memory usage: 634.5 KB


In [12]:
customers.head()


,customer_id,first_name,last_name,email,phone,city,state,region,customer_tier,signup_date
0,C00001,Michael,Robinson,michael.robinson1@example.com,555-496-7825,Denver,CO,West,Gold,2025-02-04
1,C00002,Michael,Smith,michael.smith2@example.com,555-307-8259,Atlanta,GA,South,Silver,2025-07-21
2,C00003,Sophia,Harris,sophia.harris3@example.com,555-659-4218,San Francisco,CA,West,Silver,2023-08-26
3,C00004,Amelia,Garcia,amelia.garcia4@example.com,555-291-3847,Philadelphia,PA,Northeast,Standard,2024-02-12
4,C00005,Robert,Wilson,robert.wilson5@example.com,555-669-5329,Atlanta,GA,South,Silver,2022-11-26


In [23]:
# Number of Customers by region
customers_by_region = customers.groupby("region")["customer_id"].count()

print(customers_by_region)

region
Midwest      1241
Northeast    1203
South        2863
West         2813
Name: customer_id, dtype: int64


# Employees

In [ ]:
employees.info()

# hire_date is a string -- we will need to change that downstream
# An employee is tied to a specific store_id
# evaluate if any employees belong to more than one store

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   employee_id  300 non-null    str  
 1   first_name   300 non-null    str  
 2   last_name    300 non-null    str  
 3   role         300 non-null    str  
 4   store_id     300 non-null    str  
 5   city         300 non-null    str  
 6   state        300 non-null    str  
 7   hire_date    300 non-null    str  
dtypes: str(8)
memory usage: 18.9 KB


In [ ]:
employees.head()

,employee_id,first_name,last_name,role,store_id,city,state,hire_date
0,E0001,Sofia,Thompson,Sales Lead,S005,Seattle,WA,2022-11-13
1,E0002,Evelyn,Brown,Assistant Manager,S014,New York,NY,2019-12-18
2,E0003,Christopher,Lopez,Assistant Manager,S011,Chicago,IL,2024-02-09
3,E0004,Grace,Thomas,Assistant Manager,S016,Dallas,TX,2020-02-04
4,E0005,Amelia,Hernandez,Sales Associate,S010,Seattle,WA,2019-07-03


In [28]:
employee_by_store = employees.groupby(["employee_id", "store_id"])["store_id"].count()
employees
print(employee_by_store)

# Found that some employees (E0298) are not assigned to a store_id
# I am curious of two things: the hire date of the employees (could it be relatively recent where they have not been assigned)
# Or what their role is -- if they are corporate and/or interns, they may not be assigned to a single store
print("--------------------------")

employees_w_unknown_store = employees[employees["store_id"] == "UNKNOWN"]
print(employees_w_unknown_store)

employee_id  store_id
E0001        S005        1
E0002        S014        1
E0003        S011        1
E0004        S016        1
E0005        S010        1
                        ..
E0296        S022        1
E0297        UNKNOWN     1
E0298        UNKNOWN     1
E0299        S004        1
E0300        S017        1
Name: store_id, Length: 300, dtype: int64
--------------------------
    employee_id first_name  last_name               role store_id  \
28        E0029     Sophia      Moore    Sales Associate  UNKNOWN   
61        E0062      Brian      Moore         sales lead  UNKNOWN   
72        E0073       Ella   Robinson      Store Manager  UNKNOWN   
98        E0099      Grace     Wilson  Assistant Manager  UNKNOWN   
109       E0110    Michael      Perez  Assistant Manager  UNKNOWN   
119       E0120     Hannah     Martin         Sales Lead  UNKNOWN   
132       E0133    Abigail      Moore  Assistant Manager  UNKNOWN   
164       E0165     Joshua      Lopez   Customer Service  UN

# Found no correlation to UNKNOWN store_id of my two hypotheses above
# I want analyze the list of all job titles (data quality check)

In [ ]:
role_list = employees.role
print(role_list.drop_duplicates())

# Found the naming inconsistencies
# Will address with standardization

0            Sales Lead
1     Assistant Manager
4       Sales Associate
6         Store Manager
7      Customer Service
8      customer service
12      sales associate
17        store manager
27           sales lead
62    assistant manager
Name: role, dtype: str


# Order Items

In [ ]:
order_items.info()
# Might convert Dtypes to be decimal (int64/float64 variables) since we are calculating a financial value (line_revenue)

<class 'pandas.DataFrame'>
RangeIndex: 123699 entries, 0 to 123698
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      123699 non-null  str    
 1   product_id    123699 non-null  str    
 2   quantity      123699 non-null  int64  
 3   unit_price    123699 non-null  float64
 4   discount_pct  123699 non-null  float64
 5   line_revenue  123699 non-null  float64
dtypes: float64(3), int64(1), str(2)
memory usage: 5.7 MB


In [24]:
order_items.head()

# Want to look for duplicate order_id/product_id combinations
# a single order_id should NOT have more than one line for the same product_id

orders_w_duplicate_product_ids = order_items.duplicated(subset=["order_id", "product_id"], keep=False)

final_list = order_items[orders_w_duplicate_product_ids]
print(len(final_list))
print("----------------------")

# Want to see if the duplicate lines (1007 count) are different lines
final_list.sort_values(["order_id", "product_id"], ascending=False)

# Found multiple line items where a single order_id, will have multiple records of a single product_id (rather than have an increased quantity)
# I intend on consolidating these, but we could guess that maybe these were orders placed after the initial order submissiong and were given the option to include it in the same order delivery
# (Check the data dictionary for if the above is just a a possible scenario)



1007
----------------------


,order_id,product_id,quantity,unit_price,discount_pct,line_revenue
123257,O049917,P0104,1,474.03,0.20,379.22
123635,O049917,P0104,1,474.03,0.20,379.22
123166,O049878,P0313,1,553.45,0.05,525.78
123450,O049878,P0313,1,553.45,0.05,525.78
122725,O049699,P0353,2,634.10,0.20,1014.56
...,...,...,...,...,...,...
123651,O000146,P0456,4,487.60,0.15,1657.84
233,O000098,P0317,2,212.58,0.10,382.64
123638,O000098,P0317,2,212.58,0.10,382.64
86,O000032,P0045,2,117.04,0.20,187.26


In [ ]:
dupe_order_items = order_items[order_items.duplicated(subset=(["order_id", "product_id"]), keep=False)].sort_values("order_id")

dupe_order_items.head(10)

# Some of them appear like true duplicates, where as others have different quantity, discount prices

,order_id,product_id,quantity,unit_price,discount_pct,line_revenue
86,O000032,P0045,2,117.04,0.20,187.26
87,O000032,P0045,1,117.04,0.10,105.34
233,O000098,P0317,2,212.58,0.10,382.64
123638,O000098,P0317,2,212.58,0.10,382.64
357,O000146,P0456,4,487.60,0.15,1657.84
360,O000146,P0465,1,69.18,0.00,69.18
123655,O000146,P0465,1,69.18,0.00,69.18
123651,O000146,P0456,4,487.60,0.15,1657.84
937,O000378,P0217,1,153.35,0.00,153.35
941,O000378,P0217,3,153.35,0.05,437.05


In [36]:
# Count exact duplicate rows within the repeated combinations
dupe_order_items.duplicated().sum()

np.int64(278)

In [37]:
# Identify rows that are exact duplicates across all columns
exact_dupes = dupe_order_items[
    dupe_order_items.duplicated(keep=False)
].sort_values(["order_id", "product_id"])
exact_dupes.head(10)

,order_id,product_id,quantity,unit_price,discount_pct,line_revenue
233,O000098,P0317,2,212.58,0.10,382.64
123638,O000098,P0317,2,212.58,0.10,382.64
357,O000146,P0456,4,487.60,0.15,1657.84
123651,O000146,P0456,4,487.60,0.15,1657.84
360,O000146,P0465,1,69.18,0.00,69.18
123655,O000146,P0465,1,69.18,0.00,69.18
1371,O000549,P0037,1,148.68,0.10,133.81
123497,O000549,P0037,1,148.68,0.10,133.81
123694,O000634,P0040,3,268.87,0.10,725.95
1573,O000634,P0040,3,268.87,0.10,725.95


Duplicate Conditions:
- drop_duplicates() : keeps one copy of each identical row
- .copy() creates a separate DataFrame

In [38]:
# Removing exact duplicates
order_items_deduped = order_items.drop_duplicates().copy()

In [ ]:
print("Original rows:", len(order_items))
print("Rows after deduplication:", len(order_items_deduped))
print("Remaining exact duplicates:", order_items_deduped.duplicated().sum())
print("Repeated order/product combinations:", order_items_deduped.duplicated(subset=["order_id", "product_id"]).sum())

Original rows: 123699
Rows after deduplication: 123421
Remaining exact duplicates: 0
Repeated orde/product combinations: 226


In [41]:
remaining_dupes = order_items_deduped[
    order_items_deduped.duplicated(
        subset=["order_id", "product_id"],
        keep=False
    )
].sort_values(["order_id", "product_id"])

remaining_dupes.head(20)

,order_id,product_id,quantity,unit_price,discount_pct,line_revenue
86,O000032,P0045,2,117.04,0.20,187.26
87,O000032,P0045,1,117.04,0.10,105.34
937,O000378,P0217,1,153.35,0.00,153.35
941,O000378,P0217,3,153.35,0.05,437.05
973,O000391,P0351,2,370.44,0.20,592.70
974,O000391,P0351,2,370.44,0.05,703.84
1861,O000761,P0316,2,249.34,0.10,448.81
1862,O000761,P0316,1,249.34,0.00,249.34
2787,O001129,P0227,1,366.96,0.10,330.26
2789,O001129,P0227,2,366.96,0.15,623.83


In [42]:
order_items_deduped["expected_revenue"] = (
    order_items_deduped["quantity"]
    * order_items_deduped["unit_price"]
    * (1 - order_items_deduped["discount_pct"])
)

In [43]:
order_items_deduped[
    ["quantity", "unit_price", "discount_pct",
     "line_revenue", "expected_revenue"]
].head()

,quantity,unit_price,discount_pct,line_revenue,expected_revenue
0,1,571.55,0.10,514.40,514.3950
1,1,389.84,0.00,389.84,389.8400
2,3,544.71,0.00,1634.13,1634.1300
3,1,170.99,0.15,145.34,145.3415
4,1,502.60,0.15,427.21,427.2100


In [44]:
# Checking for zero or negative quantities
order_items_deduped[order_items_deduped["quantity"] <= 0]

,order_id,product_id,quantity,unit_price,discount_pct,line_revenue,expected_revenue
63,O000024,P0181,0,274.89,0.00,274.89,0.0000
870,O000351,P0370,-1,607.70,0.15,516.55,-516.5450
2115,O000858,P0374,0,184.98,0.00,184.98,0.0000
2394,O000963,P0222,0,518.00,0.20,414.40,0.0000
4078,O001656,P0266,-1,72.29,0.15,61.45,-61.4465
...,...,...,...,...,...,...,...
117961,O047726,P0230,0,155.07,0.00,310.14,0.0000
119108,O048205,P0163,-1,329.96,0.05,313.46,-313.4620
120109,O048615,P0385,-1,310.60,0.00,1242.40,-310.6000
120451,O048749,P0010,0,461.52,0.00,461.52,0.0000


In [46]:
# Check for discounts outside the range 0 to 1
order_items_deduped[(order_items_deduped["discount_pct"] < 0) | (order_items_deduped["discount_pct"] > 1)]

,order_id,product_id,quantity,unit_price,discount_pct,line_revenue,expected_revenue
662,O000264,P0165,4,600.15,-0.1,2400.60,2640.660
1812,O000741,P0304,1,186.31,1.2,186.31,-37.262
14524,O005958,P0080,1,437.48,1.2,437.48,-87.496
17023,O006941,P0134,2,92.76,-0.1,176.24,204.072
19308,O007861,P0311,2,419.98,1.2,839.96,-167.992
20133,O008204,P0189,1,17.17,-0.1,17.17,18.887
25976,O010500,P0318,4,330.39,1.2,1321.56,-264.312
29987,O012125,P0400,2,323.87,-0.1,647.74,712.514
35340,O014255,P0022,1,412.31,1.2,329.85,-82.462
39473,O015948,P0299,1,227.99,-0.1,227.99,250.789


# Orders Analysis

In [8]:
orders.info()
# 50100 orders but only 49980 store_id
# I'll investigate if they are associated to a channel of online by chance or if the order status is not complete


<class 'pandas.DataFrame'>
RangeIndex: 50100 entries, 0 to 50099
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        50100 non-null  str  
 1   customer_id     50100 non-null  str  
 2   order_date      50100 non-null  str  
 3   store_id        49980 non-null  str  
 4   employee_id     27502 non-null  str  
 5   channel         50100 non-null  str  
 6   payment_method  50100 non-null  str  
 7   order_status    50100 non-null  str  
dtypes: str(8)
memory usage: 3.1 MB


In [38]:
orders.head()
# Already see one example of a NaN employee is associated to a channel of Online
# store_id w/ NaN is associated to channel = store


,order_id,customer_id,order_date,store_id,employee_id,channel,payment_method,order_status
0,O000001,C04331,2026-04-14 21:39:23,S010,E0199,Mobile App,Apple Pay,Completed
1,O000002,C00730,2025-07-21 15:39:25,S006,E0055,Store,Gift Card,Completed
2,O000003,C04026,2024-09-21 00:31:10,NaN,E0074,Store,PayPal,Completed
3,O000004,C06548,2024-01-02 09:29:03,S013,E0058,Mobile App,PayPal,Completed
4,O000005,C07152,2025-04-22 04:34:35,S028,NaN,Online,PayPal,Completed


In [ ]:
# I want to look at if employee_id = NaN, all have a channel of Online

nan_employees = orders[orders["employee_id"].isna()]
#print(nan_employees)

nan_employees_by_channel = nan_employees.groupby("channel")["employee_id"].size()
print(nan_employees_by_channel)

# All of the employees tied to NaN are assigned to Online and/or online

channel
Online    22550
online       48
Name: employee_id, dtype: int64


In [ ]:
#orders["order_status"].unique()
# Completed, Cancelled, Pending

#orders["channel"].unique()
# Mobile App, Store, Online, online

orders["payment_method"].unique()
# Apple Pay, Gift Card, PayPal, Credit Card, Debit Card

<StringArray>
['Apple Pay', 'Gift Card', 'PayPal', 'Credit Card', 'Debit Card']
Length: 5, dtype: str

In [22]:
# Look for duplicate orders
dupes = orders[orders.duplicated(subset="order_id", keep=False)].sort_values("order_id")
dupes.head(10)
# 200 duplicate orders (by order_id)


,order_id,customer_id,order_date,store_id,employee_id,channel,payment_method,order_status
588,O000589,C06384,2025-03-28 04:00:15,S012,E0121,Mobile App,Gift Card,Completed
50034,O000589,C06384,2025-03-28 04:00:15,S012,E0121,Mobile App,Gift Card,Completed
50001,O001164,C04233,2025-09-07 18:58:30,S009,E0264,Store,PayPal,Completed
1163,O001164,C04233,2025-09-07 18:58:30,S009,E0264,Store,PayPal,Completed
1231,O001232,C07371,2024-10-31 20:42:45,S005,NaN,Online,PayPal,Completed
50087,O001232,C07371,2024-10-31 20:42:45,S005,NaN,Online,PayPal,Completed
1571,O001572,C03195,2024-04-15 01:13:40,S010,NaN,Online,Credit Card,Pending
50011,O001572,C03195,2024-04-15 01:13:40,S010,NaN,Online,Credit Card,Pending
50079,O003557,C07607,2025-11-19 06:02:00,S019,E0079,Store,PayPal,Completed
3556,O003557,C07607,2025-11-19 06:02:00,S019,E0079,Store,PayPal,Completed


# Data Cleaning

## Orders

### Rule 1: Remove exact duplicate records

**Issue:** The source contains 200 duplicate rows. All columns match for the duplicated records.

**Decision:** Retain one copy of each exact record using pandas `drop_duplicates()`.

**Validation:** Row count decreased from 50,100 to 50,000; exact duplicate count decreased to 0; unique order IDs equal 50,000.

**Limitation:** This rule removes exact duplicate rows only. It does not resolve conflicting records that share an order ID.

In [23]:
orders_clean = orders.drop_duplicates().copy()

print("Original rows: ", len(orders))
print("Cleaned rows: ", len(orders_clean))
print("Remaining exact duplicates: ", orders_clean.duplicated().sum())
print("Unique order IDs: ", orders_clean["order_id"].nunique())

Original rows:  50100
Cleaned rows:  50000
Remaining exact duplicates:  0
Unique order IDs:  50000


# Products

In [ ]:
products.info()
# 35 missing records for unit cost
# may be able to make an inference on price based on product status, category, or subcategory

<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   product_id      480 non-null    str    
 1   product_name    480 non-null    str    
 2   category        480 non-null    str    
 3   subcategory     480 non-null    str    
 4   list_price      480 non-null    float64
 5   unit_cost       445 non-null    float64
 6   product_status  480 non-null    str    
dtypes: float64(2), str(5)
memory usage: 26.4 KB


In [49]:
products.head()

,product_id,product_name,category,subcategory,list_price,unit_cost,product_status
0,P0001,Pro Portable Speaker,Electronics,Wireless Audio,27.96,14.51,Active
1,P0002,Pro Wireless Headphones,Electronics,Wireless Audio,443.73,328.89,Active
2,P0003,Urban Portable Speaker,Electronics,Wireless Audio,32.28,14.65,Active
3,P0004,Select Portable Speaker,Electronics,Wireless Audio,28.93,14.22,Discontinued
4,P0005,Smart Wireless Headphones,Electronics,Wireless Audio,387.95,275.99,Active


In [51]:
# Products by Cateogry and Subcategory
category_sub_pivot = products.groupby(["category", "subcategory"])["list_price"].mean()
print(category_sub_pivot)

category           subcategory         
Apparel            Accessories             365.898333
                   Footwear                202.478125
                   Men's Apparel           343.898750
                   Women's Apparel         298.125882
Beauty             Cosmetics               430.245000
                   Haircare                356.262353
                   Personal Care           370.252500
                   Skincare                321.488125
Electronics        Computer Accessories    332.447222
                   Mobile Devices          403.894737
                   Smart Home              214.523333
                   Wireless Audio          310.964118
Home               Bedding                 354.230625
                   Decor                   334.974737
                   Furniture               333.511579
                   Kitchen                 343.510556
Office             Desk Accessories        393.719474
                   Office Furniture       

In [3]:
products["category"].unique()

# Each category has a lowercase category name of it

<StringArray>
[      'Electronics',       'electronics',              'Home',
              'home',           'Apparel',           'apparel',
 'Sports & Outdoors', 'sports & outdoors',            'beauty',
            'Beauty',            'Office',            'office']
Length: 12, dtype: str

# TO-DO: additional deep dive on the products subcategory
# Look into min and max unit cost per category

In [ ]:
products["unit_cost"].min() #7.9
products["unit_cost"].max() # 489.74

np.float64(489.74)

In [ ]:
min_max_cost_category = products.groupby("category")["unit_cost"].agg(["min", "max", "mean", "median"])

sorted_mmcc = min_max_cost_category.sort_values(["mean"], ascending=False)
print(sorted_mmcc)

# Hard to analyze since their are duplicates of each category


                     min     max        mean   median
category                                             
apparel            85.54  338.91  248.473846  254.900
home               32.12  403.32  237.243750  228.895
electronics        36.04  408.19  236.070000  241.440
sports & outdoors  98.23  370.45  226.975000  215.300
Beauty              7.90  449.74  223.894032  214.330
office             34.82  482.31  215.384000  221.035
Office              9.18  489.74  213.874407  195.450
Home                8.61  394.26  200.004242  207.820
beauty             52.54  461.41  199.051667  167.665
Apparel            11.66  482.93  197.297097  191.500
Sports & Outdoors  19.89  422.61  191.155303  173.705
Electronics        14.22  436.86  185.904444  192.755


# Returns

In [ ]:
returns.info()

# Nothing obvious here
# We can investigate the return_id and see if their are duplicate

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   return_id        8000 non-null   str  
 1   order_id         8000 non-null   str  
 2   product_id       8000 non-null   str  
 3   return_quantity  8000 non-null   int64
 4   return_reason    8000 non-null   str  
 5   return_date      8000 non-null   str  
dtypes: int64(1), str(5)
memory usage: 375.1 KB


# Stores

In [59]:
stores.info()
# Nothing obvious with stores

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   store_id    20 non-null     str  
 1   store_name  20 non-null     str  
 2   city        20 non-null     str  
 3   state       20 non-null     str  
 4   region      20 non-null     str  
 5   store_type  20 non-null     str  
dtypes: str(6)
memory usage: 1.1 KB


In [60]:
stores.head()

,store_id,store_name,city,state,region,store_type
0,S001,New York Main,New York,NY,Northeast,Urban
1,S002,Boston Market,Boston,MA,Northeast,Urban
2,S003,Philadelphia Market,Philadelphia,PA,Northeast,Mall
3,S004,Chicago Central,Chicago,IL,Midwest,Mall
4,S005,Detroit Main,Detroit,MI,Midwest,Suburban


In [ ]:
stores.groupby("store_id")["store_name"].count()

# Each store_id has only one store name associated with it

store_id
S001    1
S002    1
S003    1
S004    1
S005    1
S006    1
S007    1
S008    1
S009    1
S010    1
S011    1
S012    1
S013    1
S014    1
S015    1
S016    1
S017    1
S018    1
S019    1
S020    1
Name: store_name, dtype: int64